# MW-AeroFusion-Net on Kaggle (fallback GPU, PROGRESS §7)

Each team member runs **their own** assigned experiments on **their own** Kaggle account (don't share accounts).

1. Settings → Accelerator: **GPU T4 x2** or **P100**; Internet: **on**.
2. Add the private Kaggle Dataset `aerofusion-processed-v1` (see `ml/kaggle/README.md`) as input.
3. Set `CONFIG` below to your experiment and run all. Results land in `/kaggle/working/runs/`; download the run folder and commit its `metrics.json` + `config.yaml` from the repo machine.

In [ ]:
CONFIG = 'ml/configs/exp/m0_gru.yaml'   # your assigned experiment
SEEDS = [0]
REPO = 'https://github.com/Shashankkaranamr/AeroFusion-MP.git'

In [ ]:
!git clone -q $REPO /kaggle/working/AeroFusion-MP
%cd /kaggle/working/AeroFusion-MP
!pip install -q -r ml/requirements.txt onnxscript 2>&1 | tail -1
!git log --oneline -1

In [ ]:
# link the uploaded processed data where the code expects it
!mkdir -p ml/data/processed
!ln -sfn /kaggle/input/aerofusion-processed-v1/dataset_v1.zarr ml/data/processed/dataset_v1.zarr
!ln -sfn /kaggle/input/aerofusion-processed-v1/grid_v1.zarr ml/data/processed/grid_v1.zarr
!ln -sfn /kaggle/input/aerofusion-processed-v1/cams_fc_v1.zarr ml/data/processed/cams_fc_v1.zarr
!cp /kaggle/input/aerofusion-processed-v1/SPLITS.lock ml/data/SPLITS.lock
!pytest ml/tests -q -W ignore 2>&1 | tail -2

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: enable the accelerator')

In [ ]:
seeds = ' '.join(map(str, SEEDS))
!python ml/scripts/run_ladder.py --configs $CONFIG --seeds $seeds

In [ ]:
!mkdir -p /kaggle/working/runs && cp -r ml/runs/2* /kaggle/working/runs/ && ls /kaggle/working/runs